In [4]:
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, roc_curve, auc, precision_recall_curve
import numpy as np
from peptigraph.core.protbert_sequence_classifier import ProtBertSequenceClassifier

import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [5]:
tokenizer = BertTokenizer.from_pretrained("Rostlab/prot_bert", do_lower_case=False)
model = BertModel.from_pretrained("Rostlab/prot_bert")
model = model.to(device)
model.eval()

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30, 1024, padding_idx=0)
    (position_embeddings): Embedding(40000, 1024)
    (token_type_embeddings): Embedding(2, 1024)
    (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-29): 30 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=1024, out_features=1024, bias=True)
            (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.0, i

In [ ]:
classifier = ProtBertSequenceClassifier(tokenizer=tokenizer, model=model)

In [6]:
print("\nAnalyzing Grampa Dataset...")
print("-" * 50)

df_grampa = pd.read_csv('C:\Python\semestr_1\PeptiGraph\data\smiles\\grampa.csv')

# Get top 10 classes by frequency
class_counts = df_grampa['target'].value_counts()
top_10_classes = class_counts.head(10).index.tolist()
df_filtered = df_grampa[df_grampa['target'].isin(top_10_classes)]

_, df_reduced = train_test_split(df_filtered, test_size=0.1, random_state=42, stratify=df_filtered['target'])
df_reduced.to_csv('grampa_reduced.csv', index=False)

X = df_reduced['sequence'].values
y = df_reduced['target'].values

print(f"Original dataset size: {len(df_grampa)}")
print(f"Size after selecting top 10 classes: {len(df_filtered)}")
print(f"Final size (10% sample): {len(df_reduced)}")
print(f"Classes included: {sorted(np.unique(y))}")
print("\nClass distribution:")
for class_label in sorted(np.unique(y)):
    count = len(df_reduced[df_reduced['target'] == class_label])
    print(f"Class {class_label}: {count} samples")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.05, random_state=42, stratify=y)
X_train_embeddings = classifier._get_embeddings(X_train)
X_test_embeddings = classifier._get_embeddings(X_test)

clf = RandomForestClassifier(n_estimators=400, max_depth=20, random_state=42)
clf.fit(X_train_embeddings, y_train)
y_pred = clf.predict(X_test_embeddings)
y_pred_proba = clf.predict_proba(X_test_embeddings)

# Calculate metrics
n_classes = len(np.unique(y))
fpr = {}
tpr = {}
roc_auc = {}
precision = {}
recall = {}
pr_auc = {}

for i, class_label in enumerate(sorted(np.unique(y))):
    fpr[i], tpr[i], _ = roc_curve(y_test == class_label, y_pred_proba[:, i])
    roc_auc[i] = auc(fpr[i], tpr[i])
    
    precision[i], recall[i], _ = precision_recall_curve(y_test == class_label, y_pred_proba[:, i])
    pr_auc[i] = auc(recall[i], precision[i])
    
    print(f"\nClass {class_label}:")
    print(f"AUROC: {roc_auc[i]:.3f}")
    print(f"AUPRC: {pr_auc[i]:.3f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

macro_roc_auc = np.mean(list(roc_auc.values()))
macro_pr_auc = np.mean(list(pr_auc.values()))
print(f"\nMacro-averaged AUROC: {macro_roc_auc:.3f}")
print(f"Macro-averaged AUPRC: {macro_pr_auc:.3f}")

if device.type == 'cuda':
    torch.cuda.empty_cache()

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.



Analyzing Grampa Dataset...
--------------------------------------------------
Original dataset size: 6760
Size after selecting top 10 classes: 6760
Final size (10% sample): 676
Classes included: [0, 1]

Class distribution:
Class 0: 325 samples
Class 1: 351 samples

Class 0:
AUROC: 0.863
AUPRC: 0.880

Class 1:
AUROC: 0.863
AUPRC: 0.851

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.62      0.74        16
           1       0.74      0.94      0.83        18

    accuracy                           0.79        34
   macro avg       0.82      0.78      0.79        34
weighted avg       0.82      0.79      0.79        34


Macro-averaged AUROC: 0.863
Macro-averaged AUPRC: 0.866
